# 第 28 章：多因子模型（Multi-Factor Models）

本 Notebook 复现正文第 8–11 节：同日缩尾、z 分数与正态分数，行业与规模中性化及其与线性约束投影的关系，两个相关信号的合成 IC 与正交化顺序，缺失因子，标准化的前视，多元化；然后在构造面板上看预处理逐步恢复 IC、合成与调参、约束，并读已发布的因子收益相关。

- 手算例与构造面板（种子 20261121）是教学构造，已知数据生成过程；French 五因子、10 行业与 FF3 是 2026-10-05 冻结的已修订历史快照（聚合收益，不是个股，不是成交）。
- 核心函数来自 `src/quant_course/factor_preprocessing.py`、`factor_research.py`、`portfolio_construction.py`；章节 CLI 在 `code/ch28_multifactor.py`。本 Notebook 不写文件，须从第一个单元顺序运行。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch28_multifactor_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 缩尾、z 分数、正态分数（8.1）

In [ ]:
import numpy as np
import pandas as pd
from quant_course.factor_preprocessing import normal_scores, winsorize_cross_section, zscore_cross_section

x = pd.DataFrame([[1.0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 100]], index=pd.period_range("2026-06", periods=1, freq="M"))
w = winsorize_cross_section(x, lower=0.1, upper=0.9, min_names=10)
table = pd.DataFrame({"raw": x.iloc[0], "winsorized": w.iloc[0], "z raw": zscore_cross_section(x).iloc[0], "z winsorized": zscore_cross_section(w).iloc[0], "normal score": normal_scores(x).iloc[0]})
assert w.iloc[0].tolist() == [2, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10]
table.round(3)

## 2. 行业中性化与线性约束投影（8.2、8.3）

In [ ]:
from quant_course.factor_preprocessing import neutralize
from quant_course.portfolio_construction import project_to_constraints

cols = list("abcdef")
sig = pd.DataFrame([[1.0, -1.0, 0.0, 11.0, 10.0, 9.0]], index=pd.period_range("2026-06", periods=1, freq="M"), columns=cols)
ret = np.array([0.05, 0.03, 0.04, -0.03, -0.04, -0.05])
res = neutralize(sig, groups=pd.Series(list("AAABBB"), index=cols))
A = np.array([[1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 1]], dtype=float)
proj = project_to_constraints(sig.iloc[0].to_numpy(), A)
assert np.allclose(proj, res.iloc[0], atol=1e-12)
print("raw IC", round(np.corrcoef(sig.iloc[0], ret)[0, 1], 4), "| neutral IC", round(np.corrcoef(res.iloc[0], ret)[0, 1], 4))
pd.DataFrame({"residual": res.iloc[0], "projection": proj}).round(4)

## 3. 两个相关信号：合成与顺序（8.4）

In [ ]:
from quant_course.factor_preprocessing import composite_ic, optimal_composite_weights, sequential_orthogonal_ic

R = np.array([[1.0, 0.5], [0.5, 1.0]]); rho = [0.04, 0.02]
weights, best = optimal_composite_weights(rho, R)
out = {"only z1": 0.04, "equal weights": composite_ic(rho, R), "z1 then z2": sequential_orthogonal_ic(rho, R, [0, 1]),
       "z2 then z1": sequential_orthogonal_ic(rho, R, [1, 0]), "optimal": best}
assert out["z2 then z1"] > out["equal weights"] > out["z1 then z2"] and abs(best - 0.04) < 1e-12
pd.Series(out).round(4)

## 4. 缺失因子与标准化的前视（8.5、8.6）

In [ ]:
from quant_course.factor_preprocessing import combine_signals, expanding_zscore

idx = pd.period_range("2026-06", periods=1, freq="M")
z = {"f1": pd.DataFrame([[2.0, np.nan]], index=idx, columns=["s1", "s2"]), "f2": pd.DataFrame([[2.0, 1.0]], index=idx, columns=["s1", "s2"])}
rw, _ = combine_signals(z, {"f1": 0.5, "f2": 0.5}, missing="reweight"); zf, _ = combine_signals(z, {"f1": 0.5, "f2": 0.5}, missing="zero")
s = pd.DataFrame({"x": [4.0, 6.0, 8.0, 20.0]}, index=pd.period_range("2026-01", periods=4, freq="M"))
past = expanding_zscore(s, min_periods=3)["x"].iloc[2]; full = ((s["x"] - s["x"].mean()) / s["x"].std(ddof=1)).iloc[2]
assert rw.iloc[0, 1] == 1.0 and zf.iloc[0, 1] == 0.5 and past > 0 > full
print("single-factor name: reweighted", rw.iloc[0, 1], "| zero-filled", zf.iloc[0, 1], "| third obs: expanding z", round(past, 3), "full-sample z", round(full, 3))

## 5. 构造面板：预处理逐步恢复 IC、合成与调参、约束（10 节）

In [ ]:
ch28 = load_code_module("ch28_multifactor.py")
cons = ch28.constructed_study()
steps = pd.DataFrame({f: {k: v["mean"] for k, v in cons["factor_ic"][f].items() if isinstance(v, dict)} for f in ch28.FACTORS}).T
assert steps.loc["momentum", "raw_pearson"] < steps.loc["momentum", "winsor_pearson"] < steps.loc["momentum", "neutral_winsor_pearson"]
steps[["raw_pearson", "winsor_pearson", "neutral_winsor_pearson", "neutral_winsor_spearman", "true_signal_pearson"]].round(4)

In [ ]:
att = pd.DataFrame(cons["tuning"]["attenuation"]).T[["true", "ratio_raw_to_true", "ratio_winsor_to_true", "ratio_neutral_winsor_to_true"]]
print("predicted raw/true (section 6.2): value %.3f, momentum %.3f, quality %.3f" % tuple(1 / np.sqrt(1 + 0.9 + c * c + 1.28) for c in (-0.5, 0.0, 0.3)))
att.round(3)

In [ ]:
comp = pd.DataFrame({k: {"Spearman": v["spearman"]["mean"], "Pearson": v["pearson"]["mean"]} for k, v in cons["composite_ic"].items()}).T
comp["analytic (Pearson scale)"] = [cons["composite_analytic"]["equal"], cons["composite_analytic"]["oracle_loading_weights"], *cons["composite_analytic"]["sequential"].values()]
assert comp.loc["sequential V>M>Q", "Pearson"] < comp.loc["sequential Q>M>V", "Pearson"]
comp.round(4)

In [ ]:
tune = {k: v for k, v in cons["tuning"].items() if isinstance(v, dict) and "discovery_mean" in v}
print("tuned weights beat equal weights in the hold-out in", round(100 * cons["tuning"]["share_tuned_beats_equal_in_holdout"], 1), "% of", cons["tuning"]["worlds"], "worlds")
assert cons["tuning"]["share_tuned_beats_equal_in_holdout"] < 0.5
pd.DataFrame(tune).T.round(4)

In [ ]:
pd.DataFrame(cons["constraints"]).T.round(4)

## 6. 已发布的因子收益：相关与等风险平均（11 节）

In [ ]:
if RUN_FRENCH:
    real = ch28.real_study()
    print("common months:", real["common_months"], real["first_month"], "to", real["last_month"])
    pd.DataFrame(real["correlation"]).round(2)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    print("HML-RMW correlation by period:", {k: round(v, 3) for k, v in real["correlation_hml_rmw_by_period"].items()}, "| rolling 60m range:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in real["rolling_60m_hml_rmw"].items()})
    e = real["equal_risk_composite"]
    assert abs(e["composite_sharpe"] - e["formula_sharpe"]) < 0.01 and real["correlation_hml_rmw_by_period"]["1963-07..1999-12"] < 0 < real["correlation_hml_rmw_by_period"]["2000-01..2026-08"]
    pd.Series({"composite Sharpe": e["composite_sharpe"], "formula Sharpe": e["formula_sharpe"], "average component Sharpe": e["average_component_sharpe"], "average pairwise corr": e["average_pairwise_correlation"], **{f"Sharpe {k}": v for k, v in e["component_sharpe"].items()}}).round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. 自己试一试

1. 把 `ch28.R_TRUE` 的价值与质量相关改为 $+0.3$，重新运行第 5 节（需要自己改 `code/ch28_multifactor.py` 的副本），看依次正交化的 IC 范围怎样变化。
2. 在第 3 节试 $\rho_2=0.03$（其余不变），比较两个顺序，并用 28-C01 的恒等式验证差值。
3. 把 `winsorize_cross_section` 的分位数改为 5% 与 95%，看单个世界里动量的 Pearson IC 怎样变化；写下这个选择应该由谁在什么时候定。